## Matrix-free elasticity in 2D

Notebook 14 covers Phase A: the 1D weighted-quadrature rule itself. This notebook covers
what it *enables*: Phase B (`matrix_free_apply_2d`, a 2D tensor-product/Kronecker
matrix-vector product that never forms the Kronecker matrix) and Phase C
(`WQMatrixFreeStiffness`, the full elasticity stiffness action `K @ v`, computed without
ever assembling `K`).

Reference: `pymfiga`'s `iga/single_model/mechanical.py`
(`MechanicalModel.compute_stiffness_property`/`compute_mf_stiffness`) is the algorithm
Phase C ports -- with one important correction, covered in Part 4 below.

In [ ]:
import time
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt

from yeti_iga.future.bspline import (
    BSpline, BSplineSurface, ControlPointManager,
    Patch, GlobalDOFManager, PatchDOFManager, IGABasis1D,
    PatchIntegrator, Material, PlaneStress,
    matrix_free_apply_2d, WQMatrixFreeStiffness,
    SubdivisionRefiner, PRefiner,
)
from yeti_iga.future.postprocessing.plotting import plot_patches_2d

## Part 1 -- the 2D Kronecker sandwich product

`matrix_free_apply_2d(Mu, Mv, v_in, is_transpose)` applies a tensor-product operator
`Mv (x) Mu` (Kronecker product) to a vector, without ever forming that (possibly huge)
Kronecker matrix -- it's a "sandwich": reshape `v_in` as a `(nu x nv)` matrix (u varying
fastest, matching `future`'s own DOF convention throughout) and compute `Mu @ V @ Mv^T`.

Direct 2D port of `pymfiga`'s `MatrixFree.apply` (`common/numerics/operations/matrix_free.py`).

In [ ]:
rng = np.random.default_rng(0)
Mu = sp.random(6, 5, density=0.5, random_state=rng, format='csc')
Mv = sp.random(4, 3, density=0.5, random_state=rng, format='csc')
v_in = rng.standard_normal(5 * 3)

result = matrix_free_apply_2d(Mu, Mv, v_in)
expected = sp.kron(Mv, Mu) @ v_in

print('shapes: Mu', Mu.shape, ' Mv', Mv.shape, ' v_in', v_in.shape, '-> result', result.shape)
print('matches kron(Mv, Mu) @ v_in:', np.allclose(result, expected, atol=1e-12))
print('max abs diff:', np.max(np.abs(result - expected)))

The Kronecker matrix itself would be `(6*4) x (5*3)` = `24 x 15` here -- tiny, but for a
real 2D IGA problem with thousands of control points per direction it would be far too
large to ever form; `matrix_free_apply_2d` never does.

`is_transpose=True` lets the *same* `(Mu, Mv)` pair serve as the transpose operator too
(`kron(Mv, Mu).T`), without storing separately-transposed matrices -- used internally by
`WQMatrixFreeStiffness` below to reuse gather matrices as scatter matrices.

In [ ]:
v_out = rng.standard_normal(6 * 4)
result_t = matrix_free_apply_2d(Mu, Mv, v_out, is_transpose=True)
expected_t = sp.kron(Mv, Mu).T @ v_out
print('transpose matches kron(Mv, Mu).T @ v_out:', np.allclose(result_t, expected_t, atol=1e-12))

## Part 2 -- matrix-free elasticity stiffness, flat patch

`WQMatrixFreeStiffness(patch, law)` precomputes, once, both directions' `WeightedQuadrature1D`
rules and the pulled-back material+geometry tensor at every (tensor-product) WQ point --
then `apply(v)` computes `K @ v` for the assembled global stiffness matrix `K`, without ever
forming `K`. Validated against `PatchIntegrator.integrate_stiffness() @ v` (the existing,
Gauss-based, already-validated assembly).

`build_flat_patch` below places control points at the **Greville abscissae** (scaled to the
physical size), not equally spaced -- B-spline bases reproduce a linear map exactly there,
which is what makes this "flat-looking" rectangle *exactly* affine for any degree, not just
approximately so (an equally-spaced control net is mildly *non*-affine for degree >= 2,
enough to break WQ's exactness, so it is never used here or anywhere else in this notebook).

In [ ]:
def greville(degree, kv):
    n = len(kv) - degree - 1
    return np.array([np.mean(kv[i + 1:i + degree + 1]) for i in range(n)])


def build_flat_patch(degree, nbel_u, nbel_v, Lx=2.0, Ly=1.0):
    kv_u = np.array([0.] * (degree + 1) + list(np.linspace(0, 1, nbel_u + 1)[1:-1]) + [1.] * (degree + 1))
    kv_v = np.array([0.] * (degree + 1) + list(np.linspace(0, 1, nbel_v + 1)[1:-1]) + [1.] * (degree + 1))
    su, sv = BSpline(degree, kv_u), BSpline(degree, kv_v)
    nu, nv = len(kv_u) - degree - 1, len(kv_v) - degree - 1
    xs, ys = greville(degree, kv_u) * Lx, greville(degree, kv_v) * Ly

    mgr = ControlPointManager(dim=2)
    mapping = []
    for jv in range(nv):
        for iu in range(nu):
            mgr.add_point([xs[iu], ys[jv]])
            mapping.append(jv * nu + iu)

    dof_manager = GlobalDOFManager([2] * mgr.n_points)
    pdm = PatchDOFManager(2, mapping, dof_manager)
    return Patch(BSplineSurface(su, sv), mgr, mapping, [nu, nv], pdm)


def max_rel_err(patch, law, n_trials=5, seed=0):
    basis_u = IGABasis1D.build(patch.tensor.components[0], patch.tensor.components[0].degree + 1)
    basis_v = IGABasis1D.build(patch.tensor.components[1], patch.tensor.components[1].degree + 1)
    K = PatchIntegrator(patch, basis_u, basis_v, law).integrate_stiffness()
    wq_stiff = WQMatrixFreeStiffness(patch, law)
    rng = np.random.default_rng(seed)
    errs = []
    for _ in range(n_trials):
        v = rng.standard_normal(K.shape[0])
        errs.append(np.linalg.norm(wq_stiff.apply(v) - K @ v) / np.linalg.norm(K @ v))
    return max(errs)


law = PlaneStress(Material(E=210000., nu=0.3))
degree, nbel = 3, 6

patch = build_flat_patch(degree, nbel, nbel)
print(f'Greville abscissae (exactly affine): rel. error = {max_rel_err(patch, law):.3e}')

Machine precision: WQ's weights are constructed so that its point set + weights reproduce
certain integrals *exactly* for a given polynomial degree, and an affine geometry keeps the
pulled-back integrand inside that reproducible space. Greville placement is what keeps this
patch affine for any degree, so the match with `PatchIntegrator`'s Gauss-based assembly is
exact (to floating-point roundoff), not merely close.

## Part 3 -- curved geometry: NURBS quarter-ring, and a convergence check

For a genuinely curved (non-affine) geometry the pulled-back integrand becomes *rational*,
not polynomial -- WQ's finite-degree target space can no longer reproduce it exactly, so
some approximation error is expected. What matters is whether that error is a *real,
convergent* approximation (shrinking as the mesh refines) or just "close by accident" at
one mesh size -- checked here on the canonical NURBS quarter-ring (inner radius 1, outer
radius 2), same fixture as `09_quarter_ring_nurbs.ipynb` and `tests/future/test_nurbs.py`.

In [ ]:
def build_quarter_ring(degree=2, refine_u=0, refine_v=0):
    r1, r2 = 1.0, 2.0
    w_c = 1.0 / np.sqrt(2.0)
    mgr = ControlPointManager(dim=2)
    for (x, y, w) in [
        (r1, 0., 1.0), (r1, r1, w_c), (0., r1, 1.0),
        (r2, 0., 1.0), (r2, r2, w_c), (0., r2, 1.0),
    ]:
        mgr.add_point([x, y], w)

    su = BSpline(2, np.array([0., 0., 0., 1., 1., 1.]))  # angular, base degree 2 (exact circle)
    sv = BSpline(1, np.array([0., 0., 1., 1.]))            # radial, base degree 1
    mapping = list(range(6))
    dof_mgr = GlobalDOFManager([2] * 6)
    pdm = PatchDOFManager(2, mapping, dof_mgr)
    patch = Patch(BSplineSurface(su, sv), mgr, mapping, [3, 2], pdm)

    # k-refinement to a common target `degree` in both directions -- angular can't go
    # below 2 (needed for an exact circle), radial starts one degree lower by construction.
    assert degree >= 2, 'angular direction needs degree >= 2 for an exact circle'
    elevate_u, elevate_v = degree - 2, degree - 1
    if elevate_u > 0:
        PRefiner(direction=0, n_elevations=elevate_u).refine(patch)
    if elevate_v > 0:
        PRefiner(direction=1, n_elevations=elevate_v).refine(patch)
    SubdivisionRefiner(direction=0, n_levels=refine_u).refine(patch)
    SubdivisionRefiner(direction=1, n_levels=refine_v).refine(patch)
    return patch


### What this geometry actually looks like

Before sweeping degree and mesh refinement, a look at the geometry itself: the coarsest
mesh (1 element, `degree=2`, the minimum needed to represent a circle exactly) against a
refined, higher-degree instance. Circles as squares are unit-weight control points;
circles-as-circles (the plotting convention for non-unit weight, from `09_quarter_ring_nurbs.ipynb`)
mark the two mid-arc control points whose weight `1/sqrt(2)` makes the NURBS trace an
exact quarter circle -- this is the same fixture used throughout Part 3.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))

plot_patches_2d(build_quarter_ring(degree=2, refine_u=0, refine_v=0),
                show_control_points=True, show_weights=True, ax=axes[0],
                title='Coarsest mesh: degree 2, 1 element')

plot_patches_2d(build_quarter_ring(degree=4, refine_u=2, refine_v=2),
                show_control_points=True, show_weights=True, ax=axes[1],
                title='Refined: degree 4, 4x4 elements')

plt.tight_layout()
plt.show()


In [ ]:
ring_degrees = [2, 3, 4, 5]
ring_levels = [0, 1, 2, 3, 4]

ring_results = {}
for degree in ring_degrees:
    errors, ndofs = [], []
    for r in ring_levels:
        patch = build_quarter_ring(degree=degree, refine_u=r, refine_v=r)
        err = max_rel_err(patch, law)
        errors.append(err)
        basis_u = IGABasis1D.build(patch.tensor.components[0], degree + 1)
        basis_v = IGABasis1D.build(patch.tensor.components[1], degree + 1)
        ndofs.append(PatchIntegrator(patch, basis_u, basis_v, law).integrate_stiffness().shape[0])
        print(f'degree={degree}  refinement level {r}: ndof={ndofs[-1]:4d}  max rel. error = {errors[-1]:.3e}')
    ring_results[degree] = dict(ndof=ndofs, err=errors)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
colors = plt.cm.viridis(np.linspace(0, 0.85, len(ring_degrees)))

for degree, color in zip(ring_degrees, colors):
    r = ring_results[degree]
    ax.loglog(r['ndof'], r['err'], 'o-', color=color, label=f'degree {degree}')

ax.set_xlabel('number of DOFs')
ax.set_ylabel('max relative error, apply(v) vs K @ v')
ax.set_title('WQMatrixFreeStiffness convergence, curved NURBS quarter-ring')
ax.legend(fontsize=8)
ax.grid(True, which='both', alpha=0.3)
plt.show()


The error shrinks monotonically under refinement at every degree -- a genuine convergent
approximation, as expected for a rational (not polynomial) integrand -- and **convergence
clearly accelerates with degree**: the apparent convergence order (fitted from the first to
the last refinement level) goes from ~1.5 at `degree=2` up to ~2.6 at `degree=5`, and the
finest-mesh error at `degree=5` (`~2e-5`) is almost 40x smaller than at `degree=2` (`~8e-4`)
for a comparable `ndof`. This is exactly the behaviour a genuine geometric approximation
error should have: it improves with higher degree the way any IGA approximation error does,
unlike a fixed construction defect, which neither refinement nor degree elevation would fix.

## Part 4 -- a subtlety: `future`'s material-tensor convention, not `pymfiga`'s

Unlike Phases A and B (direct, faithful ports of `pymfiga`), Phase C could **not** just
reuse `pymfiga`'s own 4th-order elasticity tensor
(`C_ijkl = lambda*d_ij*d_kl + mu*(d_ik*d_jl+d_il*d_jk)`, the textbook isotropic tensor,
contracted directly with the two basis gradients). `future`'s own, already-validated
`ConstitutiveLaw::stiffness_density` -- the B-free kernel `integrate_stiffness()` actually
uses -- corresponds to a *different* tensor,
`D_IJlm = lambda*d_Il*d_Jm + mu*d_Im*d_Jl + mu*d_IJ*d_lm` (both are valid elasticity tensors
in the abstract, but they give different *local* per-basis-pair blocks -- only pymfiga's own,
separately-assembled global matrix would coincide with `future`'s). Confirmed numerically
below rather than assumed:

In [ ]:
def K_future_formula(lam, mu, v, w):
    v, w = np.asarray(v), np.asarray(w)
    return lam * np.outer(v, w) + mu * np.outer(w, v) + mu * np.dot(v, w) * np.eye(2)


def K_pymfiga_formula(lam, mu, v, w):
    v, w = np.asarray(v), np.asarray(w)
    return lam * np.dot(v, w) * np.eye(2) + mu * (np.outer(v, w) + np.outer(w, v))


mat = Material(E=210000., nu=0.3)
law_probe = PlaneStress(mat)
K_unit = np.array(law_probe.stiffness_density([1.0, 0.0], [1.0, 0.0], [0.0, 0.0]))
mu = K_unit[1, 1]
lam = K_unit[0, 0] - 2 * mu

rng = np.random.default_rng(1)
v, w = rng.standard_normal(2), rng.standard_normal(2)
K_actual = np.array(law_probe.stiffness_density(v, w, [0.0, 0.0]))

print('matches future formula   (lambda*v(x)w + mu*w(x)v + mu*(v.w)*I):',
      np.allclose(K_actual, K_future_formula(lam, mu, v, w)))
print('matches pymfiga formula  (lambda*(v.w)*I + mu*(v(x)w + w(x)v)) :',
      np.allclose(K_actual, K_pymfiga_formula(lam, mu, v, w)))

`WQMatrixFreeStiffness` extracts `lambda`/`mu` from *any* `ConstitutiveLaw` this same way --
numerically probing `stiffness_density` with unit vectors, rather than assuming a formula or
adding a new accessor to `ConstitutiveLaw` -- so it stays correct even for a future custom
isotropic law.

## Part 5 -- setup cost vs. per-application cost

Matrix-free methods trade *assembly* cost for *repeated application* cost -- worth measuring
rather than assuming. `WQMatrixFreeStiffness`'s constructor plays the role of
`integrate_stiffness()`'s assembly; `apply(v)` plays the role of a plain sparse `K @ v`:

In [ ]:
degree, nbel = 3, 30
patch = build_flat_patch(degree, nbel, nbel)
basis_u = IGABasis1D.build(patch.tensor.components[0], degree + 1)
basis_v = IGABasis1D.build(patch.tensor.components[1], degree + 1)

t0 = time.perf_counter()
K = PatchIntegrator(patch, basis_u, basis_v, law).integrate_stiffness()
t_assemble = time.perf_counter() - t0

t0 = time.perf_counter()
wq_stiff = WQMatrixFreeStiffness(patch, law)
t_setup = time.perf_counter() - t0

v = np.random.default_rng(0).standard_normal(K.shape[0])
n_rep = 30

t0 = time.perf_counter()
for _ in range(n_rep):
    Kv = K @ v
t_matvec = (time.perf_counter() - t0) / n_rep

t0 = time.perf_counter()
for _ in range(n_rep):
    Kv_mf = wq_stiff.apply(v)
t_apply = (time.perf_counter() - t0) / n_rep

print(f'ndof = {K.shape[0]}')
print(f'Gauss assembly:        {t_assemble*1e3:7.2f} ms   (one-time)')
print(f'WQ matrix-free setup:  {t_setup*1e3:7.2f} ms   (one-time)')
print(f'assembled K @ v:       {t_matvec*1e3:7.3f} ms   (per application)')
print(f'matrix-free apply(v):  {t_apply*1e3:7.3f} ms   (per application)')

The one-time **setup** is markedly cheaper matrix-free (no sparse matrix ever assembled) --
but the **per-application** cost is currently higher than a plain sparse matvec on the
already-assembled matrix: `apply()` makes 24 separate `matrix_free_apply_2d` calls (8 gathers
+ 16 scatters, one pair per `(I, J, alpha, beta)` combination) internally, each still forming
a dense intermediate, none of it yet parallelized -- unlike Eigen's highly optimized sparse
matvec. This is expected for a first, correctness-focused implementation, not yet a
performance one -- see *Next steps*. Matrix-free is worth it when *assembly itself* is the
bottleneck (very large or 3D problems where `K` may not even fit in memory), not
automatically for every matvec-bound workload.

## Next steps

- **Performance**: `apply()`'s 24 small `matrix_free_apply_2d` calls are not yet
  parallelized or fused -- there is room to cut this down (e.g. precomputing shared
  intermediate products across the `(I, J)` loop) before matrix-free applies can beat an
  assembled sparse matvec, not just assembly.
- **3D / general ND**: both `matrix_free_apply_2d` and `WQMatrixFreeStiffness` are
  hardcoded to 2 parametric directions / 2 displacement components, matching `future`'s
  current 2D-solid scope. `pymfiga`'s own `MatrixFree.apply` is already ND-generic --
  extending to 3D would mean a genuine 3-mode tensor contraction, not just adding a
  direction to the existing sandwich.
- **Matrix-free mass matrix**: `pymfiga`'s `compute_mf_mass` (same file) is the direct
  analogue, not yet ported -- much simpler than stiffness (no Jacobian pullback needed
  beyond `|detJ|` itself).
- **Iterative solver integration**: `apply()` is exactly the operator a matrix-free CG/GMRES
  solver needs (`scipy.sparse.linalg.LinearOperator`) -- not wired up yet.